<a href="https://colab.research.google.com/github/yy3462-create/textual_analysis_project/blob/main/Final_project_%22AI%22_%EF%BC%88GPE%EF%BC%89NER_to_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- CLEAN & PIN (run this first) ---
%pip install -U "pip<24.3" setuptools wheel

# Remove packages that force or prefer NumPy 2.x (not needed for this class)
%pip uninstall -y pytensor opencv-python opencv-contrib-python opencv-python-headless numba cupy-cuda12x tensorflow

# Satisfy IPython's dependency
%pip install "jedi>=0.18.0"

# Pin a NumPy that is ABI-compatible with spaCy wheels
%pip install "numpy==1.26.4"

# Now install the libraries we actually need
%pip install "spacy==3.7.4" "pandas<2.2" "networkx>=3.2" "plotly>=5.18"

print("✅ Clean install complete. NOW go to Runtime → Restart runtime, then run the next cell.")


  Using cached spacy-3.7.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (27 kB)
  Using cached weasel-0.3.4-py3-none-any.whl.metadata (4.7 kB)
Using cached spacy-3.7.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (6.5 MB)
Using cached weasel-0.3.4-py3-none-any.whl (50 kB)
  Attempting uninstall: weasel
    Found existing installation: weasel 0.4.3
    Uninstalling weasel-0.4.3:
      Successfully uninstalled weasel-0.4.3
  Attempting uninstall: spacy
    Found existing installation: spacy 3.7.5
    Uninstalling spacy-3.7.5:
      Successfully uninstalled spacy-3.7.5
✅ Clean install complete. NOW go to Runtime → Restart runtime, then run the next cell.


In [2]:
!pip install -U spacy
!pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_md-3.7.1/en_core_web_md-3.7.1-py3-none-any.whl


# Download a medium English model for better NER
!python -m spacy download en_core_web_md -q

  Using cached spacy-3.8.11-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (27 kB)
  Using cached thinc-8.3.10-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (15 kB)
  Using cached weasel-0.4.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached blis-1.3.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (7.5 kB)
Using cached spacy-3.8.11-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (33.2 MB)
Using cached thinc-8.3.10-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (3.9 MB)
Using cached weasel-0.4.3-py3-none-any.whl (50 kB)
Using cached blis-1.3.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (11.4 MB)
  Attempting uninstall: blis
    Found existing installation: blis 0.7.11
    Uninstalling blis-0.7.11:
      Successfully uninstalled blis-0.7.11
  Attempting uninstall: weasel
    Found existing installation: weasel 0.3.4
    Uninstalling weasel-0.3.4:
      Successfully uninstalled weasel-

In [4]:
import spacy
nlp = spacy.load("en_core_web_md")


In [5]:
# --- Imports (post-restart) ---
import sys, re, numpy as np, pandas as pd, spacy, networkx as nx, plotly.graph_objects as go
from itertools import combinations

# Load spaCy English model
nlp = spacy.load("en_core_web_md")
nlp.max_length = 2_000_000  # or 3_000_000 for extra headroom

# Configure pandas display
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 50)

# Verify environment
print("✅ Environment ready")
print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"spaCy: {spacy.__version__}")
print(f"pandas: {pd.__version__}")
print(f"networkx: {nx.__version__}")


✅ Environment ready
Python: 3.12.12
NumPy: 1.26.4
spaCy: 3.7.5
pandas: 2.1.4
networkx: 3.6



## 1) Load your Factiva articles

- Expect a CSV/Parquet with a **`text`** column (one article per row).  
- Optional helpful columns: `article_id`, `date`, `source`, `section`, `headline`.  
- Replace the demo data below with your real path.


In [6]:
# If using Colab, mount Drive (optional but recommended so outputs persist)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# 🔁 Replace this with your actual load (e.g., from Drive)
df = pd.read_csv("/content/drive/MyDrive/factiva_ner_project/factiva.csv")

# Demo placeholder (remove once real data is loaded)
# df = pd.DataFrame({
#     "article_id": [1,2,3],
#     "text": [
#         "President Biden met with Ursula von der Leyen in Washington to discuss trade and AI.",
#         "Elon Musk and Tim Cook appeared in Brussels at an EU competition hearing with Margrethe Vestager.",
#         "The IMF and World Bank met in Marrakech; Kristalina Georgieva spoke with Janet Yellen."
#     ],
#     "source": ["ExampleWire","ExampleWire","ExampleWire"],
#     "date": ["2024-09-10","2024-09-12","2024-10-01"]
# })
assert "Headline" in df.columns and df["Headline"].notna().all(), "Data must include a non-null 'text' column."
print(df.head(3))
print(f"Articles loaded: {len(df)}")

   index  \
0      1   
1      1   
2      1   

                                                                                                                                                   Headline  \
0                                         New to The Street Broadcasts Tonight on Bloomberg at 6:30 PM EST Featuring Roadzen, BioVie, and TY J Young Wealth   
1  AI needs power desperately. Here's how to invest in companies profiting from the pain. The shortage is a lucrative opportunity — but the window is brief   
2                                                                                          Govt announces Shs100b grants to boost climate-smart agriculture   

  Publication_Date    Source_Name  \
0       2025-12-06     ACCESSWIRE   
1       2025-12-06    MarketWatch   
2       2025-12-06  Daily Monitor   

                                                                                                                                                                     


## 2) Minimal cleaning + run NER (spaCy)

We keep cleaning light for NER (avoid over-normalizing names). We’ll batch with `nlp.pipe(...)` for speed.


In [8]:

def clean_text(s: str) -> str:
    return re.sub(r"\s+", " ", str(s)).strip()

df["text_clean"] = df["CombinedText"].map(clean_text)

def iter_docs(texts, batch_size=32):
    for doc in nlp.pipe(texts, batch_size=batch_size, disable=["lemmatizer","textcat"]):
        yield doc

docs = list(iter_docs(df["text_clean"]))
print("Docs processed:", len(docs))


Docs processed: 75



## 3) Extract entities → tidy tables

Keep `PERSON`, `ORG`, `GPE` for policy mapping; attach `article_id` for traceability.


In [9]:

KEEP = {"PERSON","ORG","GPE"}

rows = []
# Use article_id if present; else fallback to row index
if "article_id" not in df.columns:
    df["article_id"] = np.arange(1, len(df)+1)

for art_id, doc in zip(df["article_id"], docs):
    for ent in doc.ents:
        if ent.label_ in KEEP:
            rows.append({
                "article_id": art_id,
                "entity_raw": ent.text,
                "label": ent.label_,
                "start": ent.start_char,
                "end": ent.end_char
            })
ents_df = pd.DataFrame(rows)

# Normalize a little (preserve display form too)
ents_df["entity_norm"] = (ents_df["entity_raw"]
                          .str.strip()
                          .str.replace(r"\s+", " ", regex=True)
                          .str.replace(r"[’'`]", "'", regex=True))
ents_df["entity_key"] = ents_df["entity_norm"].str.lower()

# Canonical display casing (most frequent)
canonical = (ents_df.groupby("entity_key")["entity_norm"]
             .agg(lambda x: x.value_counts().idxmax())
             .rename("entity"))
ents_df = ents_df.merge(canonical, on="entity_key", how="left")

print("Entities extracted:", len(ents_df))
ents_df.head(10)


Entities extracted: 4005


,article_id,entity_raw,label,start,end,entity_norm,entity_key,entity
0,1,Laser Photonics,ORG,48,63,Laser Photonics,laser photonics,Laser Photonics
1,1,"DataVault, Aeries Technology, Sustainable Green Team",ORG,65,117,"DataVault, Aeries Technology, Sustainable Green Team","datavault, aeries technology, sustainable green team","DataVault, Aeries Technology, Sustainable Green Team"
2,1,PetVivo,ORG,119,126,PetVivo,petvivo,PetVivo
3,1,Synergy CHC NEW YORK CITY,ORG,132,157,Synergy CHC NEW YORK CITY,synergy chc new york city,Synergy CHC NEW YORK CITY
4,1,NY / ACCESS Newswire,ORG,159,179,NY / ACCESS Newswire,ny / access newswire,NY / ACCESS Newswire
5,1,https://www.accessnewswire.com/,PERSON,181,212,https://www.accessnewswire.com/,https://www.accessnewswire.com/,https://www.accessnewswire.com/
6,1,Bloomberg Television,ORG,418,438,Bloomberg Television,bloomberg television,Bloomberg Television
7,1,Roadzen,PERSON,502,509,Roadzen,roadzen,Roadzen
8,1,NASDAQ,ORG,511,517,NASDAQ,nasdaq,NASDAQ
9,1,NASDAQ,ORG,533,539,NASDAQ,nasdaq,NASDAQ



## 4) Quick QA / sanity checks

Look at frequent entities per type. Expect some noise (titles, partial names, acronyms).


In [10]:

def top_vals(df_, label, n=15):
    s = (df_[df_["label"]==label]["entity"]
         .value_counts()
         .head(n))
    print(f"\nTop {label} entities:")
    display(s)

for lab in ["PERSON","ORG","GPE"]:
    top_vals(ents_df, lab, n=10)





Top PERSON entities:


,count
entity,
Y.,29
L.,23
Li,21
J.,17
Rattlesnaq,16
Fig,15
A.,13
Haidt,12
Liu,11



Top ORG entities:


,count
entity,
AI,82
SUP,40
Apple,32
Google,23
GIF,21
Microsoft,17
NATO,14
IIMA,13
Amazon,12



Top GPE entities:


,count
entity,
India,48
U.S.,43
Ukraine,36
Pakistan,25
US,22
Taiwan,21
Australia,18
Russia,16
New Zealand,12



## 5) Build a People co‑mention network

1.   List item
2.   List item



Two people are connected if they appear **in the same article**. (Extension: connect within the same sentence for tighter links.)


In [11]:
from itertools import combinations

# 提取 GPE（地名）
gpe = ents_df[ents_df["label"]=="GPE"][["article_id","entity"]].drop_duplicates()

import re

def clean_gpe(name):
    name = name.strip()

    # 去掉笑话式的数字和奇怪的碎词
    if re.search(r"\d", name):
        return None

    # 去掉过短的垃圾字符
    if len(name) <= 2:
        return None

    return name

gpe["entity"] = gpe["entity"].apply(clean_gpe)
gpe = gpe.dropna(subset=["entity"])

edge_rows = []
for art_id, group in gpe.groupby("article_id"):
    places = sorted(group["entity"].unique())
    for a, b in combinations(places, 2):
        edge_rows.append((a,b,art_id))

edges_df = pd.DataFrame(edge_rows, columns=["src","dst","article_id"])

edge_weights = (
    edges_df.groupby(["src","dst"])
            .size()
            .reset_index(name="weight")
            .sort_values("weight", ascending=False)
)

print(edge_weights.head())
print(f"Edges (unique pairs): {len(edge_weights)} | Articles contributing: {edges_df['article_id'].nunique()}")
import networkx as nx

G = nx.Graph()

# 添加节点
for p in gpe["entity"].unique():
    G.add_node(p, type="GPE")

# 添加边
for _, row in edge_weights.iterrows():
    G.add_edge(row["src"], row["dst"], weight=int(row["weight"]))

print(f"Graph -> Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")


         src          dst  weight
2315    D.C.   Washington       6
7163  Russia      Ukraine       5
7557  Taiwan   Washington       3
6060  MANILA  Philippines       3
2305    D.C.       Taiwan       3
Edges (unique pairs): 7838 | Articles contributing: 53
Graph -> Nodes: 354, Edges: 7838


In [12]:
import re

def clean_gpe(g):
    g = g.strip()

    # 1. 去除带 "." 的非地名缩写（如 Sci., S.R., S.C., M.A.）
    if re.match(r"^[A-Za-z]\.?[A-Za-z]\.?$", g):
        return None
    if g.endswith("."):
        return None

    # 2. 单词太短的丢掉
    if len(g) <= 2:
        return None

    # 3. 单姓 Zhang / Yang / Patel 等 → 不是地名
    if re.match(r"^[A-Z][a-z]+$", g):
        # 若为常见姓氏，则剔除
        common_surnames = ["Zhang", "Yang", "Wang", "Li", "Kim", "Park", "Singh", "Patel"]
        if g in common_surnames:
            return None

    # 4. 学术缩写（SCI, MA, SR）
    if g.upper() in ["SCI", "MA", "SR", "DR"]:
        return None

    # 5. 大写缩写地名（如 USA, UAE）允许保留
    if g.isupper() and len(g) <= 3:
        return g

    # 6. 去除常见非地名词汇
    bad_tokens = ["Research", "Science", "Journal", "Corp", "Ltd", "Company"]
    if any(bt.lower() in g.lower() for bt in bad_tokens):
        return None

    return g

# 应用清洗
gpe["entity"] = gpe["entity"].apply(clean_gpe)
gpe = gpe.dropna(subset=["entity"])



## 6) Centrality: who matters? who bridges?

Degree centrality (connectivity), betweenness (bridging), and weighted degree (total co‑mentions).


In [13]:
gpe = ents_df[ents_df["label"]=="GPE"][["article_id", "entity"]].drop_duplicates()
import re

def clean_gpe(name):
    name = name.strip()

    # A. 去掉所有含 "." 的地名 —— 这一步直接杀掉 Sci. / S.R. / M.A. / S.C.
    if "." in name:
        return None

    # B. 去掉单词太短的 (一般不是地名)
    if len(name) <= 2:
        return None

    # C. 去掉常见姓氏 (避免 Zhang / Yang / Li / Wang / Park 等被误标成地名)
    surnames = {"Zhang", "Yang", "Li", "Wang", "Park", "Kim", "Singh", "Khan"}
    if name in surnames:
        return None

    # D. 必须至少含有一个字母（排除奇怪的符号）
    if not re.search(r"[A-Za-z]", name):
        return None

    return name

gpe["entity"] = gpe["entity"].apply(clean_gpe)
gpe = gpe.dropna(subset=["entity"])
from itertools import combinations

edge_rows = []
for art_id, grp in gpe.groupby("article_id"):
    places = sorted(grp["entity"].unique())
    for a, b in combinations(places, 2):
        edge_rows.append((a, b, art_id))

edges_df = pd.DataFrame(edge_rows, columns=["src", "dst", "article_id"])
edge_weights = (
    edges_df.groupby(["src","dst"])
            .size()
            .reset_index(name="weight")
            .sort_values("weight", ascending=False)
)
import networkx as nx

G = nx.Graph()

for p in gpe["entity"].unique():
    G.add_node(p)

for _, row in edge_weights.iterrows():
    G.add_edge(row["src"], row["dst"], weight=int(row["weight"]))
deg = nx.degree_centrality(G)
btw = nx.betweenness_centrality(G, normalized=True, weight="weight")
deg_w = {n: sum(d["weight"] for _,_,d in G.edges(n, data=True)) for n in G.nodes()}

cent_df = (
    pd.DataFrame({
        "entity": list(G.nodes()),
        "degree_centrality": [deg[n] for n in G.nodes()],
        "betweenness": [btw[n] for n in G.nodes()],
        "weighted_degree": [deg_w[n] for n in G.nodes()],
    })
    .sort_values(["weighted_degree","degree_centrality"], ascending=False)
    .reset_index(drop=True)
)

cent_df.head(10)


,entity,degree_centrality,betweenness,weighted_degree
0,India,0.465035,0.350963,137
1,Pakistan,0.416084,0.204287,121
2,China,0.227273,0.146579,67
3,Spain,0.216783,0.009101,63
4,Uganda,0.202797,0.000096,58
5,Holling,0.202797,0.000096,58
6,Gujrat,0.202797,0.000096,58
7,Karakoram,0.202797,0.000096,58
8,Indus,0.202797,0.000096,58
9,Britannica,0.202797,0.000096,58



## 7) Interactive network (Plotly)

Small/medium corpora render fine inline. For larger projects, export to **Gephi**.


In [14]:
# 取前 20 条最强共现关系
top20_edges = edge_weights.head(20)

# 构建 Top20 子图
G_top20 = nx.Graph()
for _, row in top20_edges.iterrows():
    G_top20.add_edge(row["src"], row["dst"], weight=row["weight"])


In [15]:
import plotly.graph_objects as go
import networkx as nx

# 布局
pos = nx.spring_layout(G_top20, k=0.8, iterations=50, seed=42)

# ---------- Edges ----------
edge_x = []
edge_y = []

for u, v in G_top20.edges():
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    mode='lines',
    line=dict(width=1.5, color='#9aa7c1'),
    hoverinfo='none'
)

# ---------- Nodes ----------
node_x = []
node_y = []
node_labels = []
node_sizes = []

# 计算 weighted degree，用于节点大小
deg_w = {n: sum(d["weight"] for _,_,d in G_top20.edges(n, data=True))
         for n in G_top20.nodes()}

for node in G_top20.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)
    node_labels.append(node)
    node_sizes.append(10 + deg_w[node] * 0.5)

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers+text',
    text=node_labels,
    textposition="top center",
    hoverinfo='text',
    marker=dict(
        size=node_sizes,
        color="#e46b6b",
        line=dict(width=1, color="white")
    )
)

# ---------- Figure ----------
fig = go.Figure(data=[edge_trace, node_trace])

fig.update_layout(
    title="Top 20 GPE Co-mention Network",
    title_x=0.5,
    width=1100,
    height=750,
    plot_bgcolor="#f5f6fa",
    margin=dict(l=20, r=20, t=60, b=20)
)

fig.show()
